# Project Summary

**Objective:** Predict whether a material is metallic using the Matbench `is_metal` dataset.

**Dataset:** `matbench_expt_is_metal` (loaded via `matminer.datasets.load_dataset`). The notebook uses composition-based MAGPIE featurization via `matminer` and `pymatgen`.

**One-line takeaway:** Tree-based models (Random Forest / Gradient Boosting) perform well without scaling; SHAP analysis highlights elemental features driving predictions and aids interpretation.

# Supervised Learning Techniques

- Decision Trees
- Random Forests
- HistGradientBoosting Classifier
- Support Vector Machines (SVM)
- Stochastic Gradient Descent based SVM

### Materials Informatics

To predict/classify if material is metallic or not using Matbench 'is_metal' dataset

Matbench is an automated leaderboard for benchmarking ML algorithm predicting a diverse range of solid materials properties

What is materials informatics?
- Use of information or features regarding materials to predict their properties
- To establish processing-structure-property-performance relationship

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score, confusion_matrix, roc_curve,auc
import shap

In [ ]:
df=load_dataset("matbench_expt_is_metal")

In [ ]:
df

In [ ]:
df["composition_obj"]=df["composition"].apply(Composition)
# creates a new column 'composition_obj' in the dataframe 'df' which has Composition objects corresponding to the chemical formulas in the 'composition' column.

In [ ]:
df.head()

In [ ]:
# Apply MAGPIE-style element features
ep_feat = ElementProperty.from_preset(preset_name="magpie")
df=ep_feat.featurize_dataframe(df,"composition_obj",ignore_errors=True) # this process adds new columns to the dataframe 'df' containing the MAGPIE-style elemental features
# it takes atleast 5 minutes to run depending on the system configuration

In [ ]:
# Final dataframe with features and target variable
X = df.drop(columns="composition_obj",inplace=True)
X=df[ep_feat.feature_labels()]
X.columns=X.columns.str.replace("MagpieData ","")

In [ ]:
X.head()

In [ ]:
X.shape

In [ ]:
y=df["is_metal"]

In [ ]:
selected_features = ['mean Electronegativity', 'mean AtomicWeight', 'mean Column', 'mean CovalentRadius']

In [ ]:
df_plot=X[selected_features].copy()
df_plot['is_metal']=y.map({True:'Metal',False:'Non-Metal'})

sns.pairplot(df_plot,hue='is_metal',plot_kws={'alpha':0.6})
plt.suptitle('Pairplot of Selected Features',y=1.02,fontsize=14)
plt.show()

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y) # split the data into training and testing sets
# stratified sampling to maintain the class distribution in both sets

In [ ]:
X_train.shape

NOTE: 
- Scaling is not necessary for tree-based models.
    - Decision trees, random forests, gradient boosting, XGBoost, and LightGBM
    Reason: They all split based on feature tresholds, not on distances or dot products
    - Tree based models are invariant to monotonic transformations like standardization or normalization (Standard Sclaer makes mean 0 and standard deviation 1)

- Scaling is needed for 
    - Support Vector Machines (SVM)
    - K-nearest neighbors (KNN)
    - Logistic regression
    - Neural Networks
    - Gaussian process models (GPR)
 

# Random Forest Classifier
- Random Forest is an ensemble method that combines multiple decision trees using bootstrap aggregation (bagging)
- Each tree is trained on a random subset of the data and features, reducing overfitting and variance

In [ ]:
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42,n_jobs=-1)
rf_classifier.fit(X_train, y_train)

In [ ]:
rf_accuracy=rf_classifier.score(X_test,y_test)
rf_accuracy

In [ ]:
y_pred=rf_classifier.predict(X_test)

### Classification Report Interpretation
- Precision: The proportion of positives that were correctly identified
- Recall (Sensitivity/TPR): The proportion of actual positives that were correctly identified
- F1-Score: The harmonic mean of precision and recall. Balances the two metrics. Useful when class distribution is imbalanced
- Support: The number of true instances of each class in the dataset

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Non-Metal', 'Metal']))

# Gradient Boosting Classifier
- Gradient boosting classifier builds trees sequentially, where each new tree learns from the residual errors of the previous one
- Unlike bagging, it focuses on reducing bias and can achieve higher accuracy but is more prone to overfitting if not properly tuned

In [ ]:
gb_classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_classifier.fit(X_train, y_train)

In [ ]:
gb_accuracy=gb_classifier.score(X_test,y_test)
gb_accuracy # observe that it is less than rf_accuracy

In [ ]:
y_pred = gb_classifier.predict(X_test)
y_proba = gb_classifier.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Non-Metal', 'Metal']))

# HistGradientBoosting Classifier

In [ ]:
hgb_classifier = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=3, random_state=42)
hgb_classifier.fit(X_train, y_train)

In [ ]:
hgb_accuracy=hgb_classifier.score(X_test,y_test)
hgb_accuracy

In [ ]:
y_pred=hgb_classifier.predict(X_test)
y_proba=hgb_classifier.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred, target_names=['Non-Metal', 'Metal']))

As the Random Forest model got higher accuracy among all now I am going to do Hyperparameter tuning to find out best estimator and best parameter values based on predefined scoring method

# Hyperparameter tuning with grid search
Hyperparameter tuning helps find the best parameters to improve accuracy, generalization, and robustness.

Here we will try to use important hyperparameters of Random forest model like number of trees, depth of trees, min samples per leaf, bootstrap etc

In [ ]:
# First we initialize model
rf_classifier=RandomForestClassifier(random_state=42)

In [ ]:
param_grid={
    'n_estimators':[100,200],
    'max_depth':[None,10,20],
    'min_samples_split':[2,5],
    'min_samples_leaf':[1,2],
    'bootstrap':[True,False]
}

In [ ]:
# Set up GridSearch
grid_search=GridSearchCV(estimator=rf_classifier,
                        param_grid=param_grid, # define parameter grid
                        cv=5, # 5-fold cross-validation
                        n_jobs=-1, # utilize all available cores
                        scoring='f1', # here we use f1 score for evaluation
                        verbose=1 # verbose output
                        )

In [ ]:
grid_search.fit(X_train,y_train)

best hyperparameters found by GridSearch

In [ ]:
grid_search.best_params_

In [ ]:
best_rf=grid_search.best_estimator_ # get the best model from grid search

In [ ]:
y_pred=best_rf.predict(X_test)
y_proba=best_rf.predict_proba(X_test)[:,1]
rf_accuracy=best_rf.score(X_test,y_test)
rf_accuracy

### Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Non-Metal', 'Metal'],
            yticklabels=['Non-Metal', 'Metal'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix: Random Forest with Hyperparameter Tuning')
plt.tight_layout()
plt.show()

## Feature importance with SHAP for Random Forest model
- Helps to understand which input features contribute the most to a model's predictions.

- SHAP (SHapley Additive exPlanations) is a model agnostic approach based on cooperative game theory.

    - Consistent: If a model relies more on feature, SHAP assigns it higher importance

    - Interpretable: It explains how each feature pushes the prediction higher or lower

In [ ]:
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(shap_values[:,:,1], X_test)

- From above SHAP plot, we can see that when 'mean NpValence' is low (blue color), it contributes positively to predicting 'Metal' (higher SHAP value).
- Conversely, when 'mean NpValence' is high (red color), it contributes negatively to predicting 'Metal' (lower SHAP value).

# Support Vector Machine (SVM) Classifier
Support vector machines are supervised learning algorithms and they work by finding the optimal hyperplane that best separates data points from different classes in a high-dimentional space. The goal is to maximize the margin between classes.

- SVMs are especially effective in high-dimensional spaces
- They can handle nonlinear classification using the kernel tricks

In [ ]:
svm_classifier = SVC(kernel='rbf', probability=True, random_state=42)
svm_classifier.fit(X_train, y_train)

In [ ]:
y_pred=svm_classifier.predict(X_test)
y_proba=svm_classifier.predict_proba(X_test)[:,1]
accuracy=svm_classifier.score(X_test,y_test)
accuracy

In [ ]:
f1_score(y_test, y_pred)

In [ ]:
roc_auc_score(y_test, y_proba)

In [ ]:
cm=confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Non-Metal', 'Metal'],
            yticklabels=['Non-Metal', 'Metal'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('SVM Confusion Matrix')
plt.tight_layout()
plt.show()

### Scaling the data using Standard Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
svm_classifier_scaled = SVC(kernel='rbf', probability=True, random_state=42)
svm_classifier_scaled.fit(X_train_scaled, y_train)

In [ ]:
y_pred=svm_classifier_scaled.predict(X_test_scaled)

In [ ]:
cm=confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Metal', 'Metal'], yticklabels=['Non-Metal', 'Metal'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('SVM with Scaling Confusion Matrix')
plt.tight_layout()
plt.show()

The model accuracy is improved after scaling which shows that the SVM model is sensitive to scaling of training data

In [ ]:
f1_score(y_test, y_pred)

In [ ]:
roc_auc_score(y_test, y_proba)

# Stochastic Gradient Descent Based SVM Classifier

In [ ]:
sgd_svm_classifer = SGDClassifier(loss="hinge", penalty="elasticnet", max_iter=1000, random_state=42)

In [ ]:
sgd_svm_classifer.fit(X_train_scaled, y_train)

In [ ]:
y_pred=sgd_svm_classifer.predict(X_test_scaled)
accuracy=sgd_svm_classifer.score(X_test_scaled,y_test)
accuracy

In [ ]:
f1_score(y_test, y_pred)

In [ ]:
roc_auc_score(y_test, y_pred)

In [ ]:
cm=confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Metal', 'Metal'], yticklabels=['Non-Metal', 'Metal'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('SVM with Scaling Confusion Matrix')
plt.tight_layout()
plt.show()

The original scaled SVM model is better than SGD-based model

## ROC and AUC Curve for scaled SVM model above 

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel("False Positive Rate",fontsize=14,fontweight='bold')
plt.ylabel("True Positive Rate",fontsize=14,fontweight='bold')
plt.title("ROC Curve for SVM Classifier", fontsize=16,fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

## Hyperparameter tuning of SVM using RandomizedSearchCV
- Faster than GridSearchCV
- Avoids overfitting by exploring a broader space with fewer evaluations
- Does not exhaustively test all possible combinations (as in GridSearchCV) it selects a random subset of combinations and evaluates using cross validations

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

In [ ]:
param_grid={
    'C': uniform(0.1, 10),
    'gamma': uniform(0.001, 1),
    'kernel': ['rbf', 'poly','sigmoid']
}

In [ ]:
svm=SVC(probability=True, random_state=42)

In [ ]:
random_search=RandomizedSearchCV(estimator=svm,
                                param_distributions=param_grid,
                                n_iter=20, # number of parameter settings that are sampled
                                cv=5, # 5-fold cross-validation
                                n_jobs=-1,
                                verbose=1,
                                scoring='f1',
                                random_state=42)

In [ ]:
random_search.fit(X_train_scaled, y_train)

In [ ]:
best_svm=random_search.best_estimator_

In [ ]:
best_params=random_search.best_params_
best_params

In [ ]:
y_pred=best_svm.predict(X_test_scaled)
y_proba=best_svm.predict_proba(X_test_scaled)[:,1]

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Non-Metal', 'Metal']))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}") 

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel("False Positive Rate",fontsize=14,fontweight='bold')
plt.ylabel("True Positive Rate",fontsize=14,fontweight='bold')
plt.title("ROC Curve for Best SVM Classifier", fontsize=16,fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()